# STAGE 1 - IMPORT AND CONFIGURATION

In [6]:
#!pip install langchain_google_genai

In [ ]:
import os
import json
import time
import asyncio
import urllib.request

import numpy as np
import pandas as pd


genai.configure(api_key = "YOUR_API_KEY_HERE")

# ---- Models -----------------------------------------------------------------
GENERATION_MODEL = "gemini-3.5-flash-lite"   # writes the RAG answers
JUDGE_MODEL      = "gemini-3.5-flash-lite"   # grades the answers
EMBEDDING_MODEL  = "gemini-embedding-001"    # turns text into vectors for retrieval

# Gemini's OpenAI-compatible endpoint (used by RAGAS in Stage 5)
GEMINI_OPENAI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

TEMPERATURE = 1.0

# ---- RAG settings -----------------------------------------------------------
CHUNK_SIZE    = 1000
CHUNK_OVERLAP = 150
TOP_K         = 3

# ---- Pricing: USD per token (Gemini 3.5 Flash-Lite: $0.30 in / $2.50 out per 1M tokens) ----
INPUT_PRICE  = 0.30 / 1_000_000
OUTPUT_PRICE = 2.50 / 1_000_000

# ---- FinanceBench: pick ONE example ----------------------------------------
FB_RAW       = "https://raw.githubusercontent.com/patronus-ai/financebench/main"
FB_JSONL_URL = f"{FB_RAW}/data/financebench_open_source.jsonl"
FINANCEBENCH_ID = "financebench_id_03029"    # 3M, FY2018 capital expenditure

DATA_DIR   = "financebench_data"
JSONL_FILE = os.path.join(DATA_DIR, "financebench_open_source.jsonl")

OUTPUT_CSV      = "FINANCEBENCH_GEMINI_EVALUATION.csv"
SUMMARY_CSV     = "FINANCEBENCH_GEMINI_EVALUATION_SUMMARY.csv"
OPERATIONAL_CSV = "FINANCEBENCH_GEMINI_OPERATIONAL_SUMMARY.csv"

print("API key loaded.")
print("Generation model:", GENERATION_MODEL)
print("Judge model     :", JUDGE_MODEL)

API key loaded.
Generation model: gemini-3.5-flash-lite
Judge model     : gemini-3.5-flash-lite


In [11]:
#downloading the files

def download(url, dest):
    """Download a file once; skip it if we already have it."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"Already downloaded: {dest}")
        return
    print(f"Downloading {url}")
    urllib.request.urlretrieve(url, dest)


os.makedirs(DATA_DIR, exist_ok=True)
download(FB_JSONL_URL, JSONL_FILE)

with open(JSONL_FILE, encoding="utf-8") as f:
    all_examples = [json.loads(line) for line in f if line.strip()]

print("Questions in the FinanceBench open-source set:", len(all_examples))
print("Fields in each example:", list(all_examples[0].keys()))

Already downloaded: financebench_data/financebench_open_source.jsonl
Questions in the FinanceBench open-source set: 150
Fields in each example: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence']


In [38]:
example = next(ex for ex in all_examples if ex["financebench_id"] == FINANCEBENCH_ID)

PDF_FILE = os.path.join(DATA_DIR, f"{example['doc_name']}.pdf")
download(f"{FB_RAW}/pdfs/{example['doc_name']}.pdf", PDF_FILE)

df = pd.DataFrame([{
    "User Query": example["question"].strip(),
    "Teacher Response": example["answer"].strip(),
}])

print("ID        :", example["financebench_id"])
print("Company   :", example["company"])
print("Document  :", example["doc_name"])
print("Question  :", df.loc[0, "User Query"])
print("Reference :", df.loc[0, "Teacher Response"])
print("Why       :", example["justification"])
print("Evidence page (0-indexed):", example["evidence"][0]["evidence_page_num"])

Already downloaded: financebench_data/3M_2018_10K.pdf
ID        : financebench_id_03029
Company   : 3M
Document  : 3M_2018_10K
Question  : What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
Reference : $1577.00
Why       : The metric capital expenditures was directly extracted from the company 10K. The line item name, as seen in the 10K, was: Purchases of property, plant and equipment (PP&E).
Evidence page (0-indexed): 59


In [39]:
df

,User Query,Teacher Response
0,What is the FY2018 capital expenditure amount ...,$1577.00


# STAGE 2 - RAG PIPELINE



In [8]:
#pip install langchain_community

In [10]:
#!pip install pypdf

In [12]:
#!pip install faiss-gpu

In [13]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

import time

/tmp/ipykernel_6064/763425851.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [14]:
#Loading the PDF
documents = PyPDFLoader(PDF_FILE).load()

#Chunking the documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks        = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))



Number of chunks: 768


In [17]:
#print(chunks[0].page_content)

In [18]:
#print(chunks[1].page_content)

In [15]:
#Creating Vector Store

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL, api_key=api_key)

# Initialize vector store with the first batch
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

In [16]:
#Model
gemini_llm = ChatGoogleGenerativeAI(model=GENERATION_MODEL, temperature=TEMPERATURE, timeout=120, max_retries=3, api_key=api_key)


In [17]:
#RAG Prompt

rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the retrieved context.

If the answer is not available in the context,
say that the information could not be found in the documents.

Context:
{context}

Question:
{question}

Answer:
""")


print("PDF pages loaded:", len(documents))
print("Chunks created:", len(chunks))
print("FAISS vector store and top-3 retriever ready.")


PDF pages loaded: 160
Chunks created: 768
FAISS vector store and top-3 retriever ready.


In [45]:
# RETRIEVE CONTEXT WITH PAGE NUMBERS

def get_context(query):
    retrieved_docs = retriever.invoke(query)
    retrieval_context = [doc.page_content for doc in retrieved_docs]
    context = "\n\n".join(retrieval_context)
    # Extract 0-indexed page numbers from metadata, defaulting to -1 if page isn't present
    pages = [doc.metadata.get("page", -1) for doc in retrieved_docs]
    return context, retrieval_context, pages

# STAGE 3 - DEFINE EVALUATION METRICS

In [24]:
#!pip install bert_score

In [26]:
#!pip install rouge_score

In [1]:
#!pip install deepeval

In [3]:
#!pip install litellm

In [19]:
from bert_score import BERTScorer
from rouge_score import rouge_scorer

from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import (
    GEval,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric
)

/tmp/ipykernel_6064/1110409804.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


In [20]:
# SIMILARITY METRICS
bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)
rouge       = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
# ANSWER QUALITY METRICS

#correctness
correctness_metric = GEval(
    name="Answer Correctness",
    model=JUDGE_MODEL,
    criteria="""
    Compare the generated answer with the reference answer.
    Evaluate factual accuracy, important information,
    and contradictions. Ignore minor wording differences.
    """,
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
    async_mode=False )

#helpfulness
helpfulness_metric = GEval(
    name="Helpfulness",
    model=JUDGE_MODEL,
    criteria="""
    Evaluate whether the answer directly addresses the question,
    is easy for a beginner to understand, and gives a clear,
    useful explanation without unnecessary information.
    """,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    async_mode=False )

#relevency
answer_relevancy_metric = AnswerRelevancyMetric(model=JUDGE_MODEL, include_reason=False, async_mode=False)

In [5]:
#!pip install ragas

In [7]:
#!pip install google-generativeai

In [9]:
#!pip install google-genai

In [22]:
#RAGAS Metrics

from openai import AsyncOpenAI
from ragas.llms import InstructorLLM
from ragas.embeddings import GoogleEmbeddings
from ragas.metrics.collections import AnswerCorrectness

import instructor
from google import genai

In [24]:
# RAGAS answer correctness
gemini_openai_client = AsyncOpenAI(api_key=api_key, base_url=GEMINI_OPENAI_BASE_URL)

ragas_llm = InstructorLLM(
    client=instructor.from_openai(gemini_openai_client, mode=instructor.Mode.JSON_SCHEMA),
    model=JUDGE_MODEL,
    provider="openai",       # "openai" here means "speaks the OpenAI protocol", which Gemini's endpoint does
    max_tokens=8192,
    temperature=TEMPERATURE,
)

# Using the modern google-genai client
ragas_embedding_client = AsyncOpenAI(api_key=api_key)
ragas_embedding_model  = GoogleEmbeddings(client=genai.Client(api_key=api_key), model=EMBEDDING_MODEL)
answer_correctness_metric = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_embedding_model)

# RAGAS GEVAL


print("RAGAS metric ready.")

RAGAS metric ready.


In [55]:
# SIMILARITY METRICS

bert_scorer = BERTScorer(lang="en", rescale_with_baseline=False)
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# ANSWER QUALITY METRICS
correctness_metric = GEval(
    name="Answer Correctness",
    model=JUDGE_MODEL,
    criteria="""
    Compare the generated answer with the reference answer.
    Evaluate factual accuracy, important information,
    and contradictions. Ignore minor wording differences.
    """,
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

helpfulness_metric = GEval(
    name="Helpfulness",
    model=JUDGE_MODEL,
    criteria="""
    Evaluate whether the answer directly addresses the question,
    is easy for a beginner to understand, and gives a clear,
    useful explanation without unnecessary information.
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    async_mode=False
)
answer_relevancy_metric = AnswerRelevancyMetric(model=JUDGE_MODEL, include_reason=False, async_mode=False)

# RAGAS ANSWER CORRECTNESS
# FIXED: Added base_url=GEMINI_OPENAI_BASE_URL to point to the Gemini OpenAI-compatible API
ragas_client = AsyncOpenAI(api_key=api_key, base_url=GEMINI_OPENAI_BASE_URL)
instructor_client = instructor.from_openai(ragas_client)
ragas_llm = InstructorLLM(
    client=instructor_client,
    model=JUDGE_MODEL,
    provider="openai",
    max_tokens=8192,
    temperature=1.0
)
ragas_embedding_client = AsyncOpenAI(api_key=api_key)
ragas_embedding_model = GoogleEmbeddings(client=ragas_embedding_client, model="text-embedding-3-small")
answer_correctness_metric = AnswerCorrectness(llm=ragas_llm, embeddings=ragas_embedding_model)

# RAG GEVAL
rag_geval_metric = GEval(
    name="RAG Correctness",
    model=JUDGE_MODEL,
    criteria="""
    Evaluate whether the generated answer is factually correct
    and consistent with the reference answer.""",
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT
    ],
    async_mode=False
)

# RETRIEVAL / RAG METRICS
context_precision_metric = ContextualPrecisionMetric(model=JUDGE_MODEL)
context_recall_metric = ContextualRecallMetric(model=JUDGE_MODEL)
faithfulness_metric = FaithfulnessMetric(model=JUDGE_MODEL)
rag_answer_relevancy_metric = AnswerRelevancyMetric(model=JUDGE_MODEL, include_reason=False, async_mode=False)
print("All evaluation metrics initialized.")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


All evaluation metrics initialized.


# STAGE 4 - INPUT QUERIES, GENERATE ANSWERS AND CALCULATE OPERATIONAL METRICS

In [46]:
from tqdm.auto import tqdm

results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="RAG generation"):

  query = row["User Query"]
  reference = row["Teacher Response"]

  try:
        total_start = time.perf_counter()

        # RETRIEVAL
        start = time.perf_counter()
        context, retrieval_context, pages = get_context(query)
        retrieval_latency = time.perf_counter() - start

        # GENERATION
        prompt_value = rag_prompt.invoke({"context": context, "question": query})
        start = time.perf_counter()
        response = gemini_llm.invoke(prompt_value)

        print(response)

        generation_latency = time.perf_counter() - start

        total_latency = time.perf_counter() - total_start

        # TOKENS AND COST
        usage = response.usage_metadata or {}
        input_tokens = usage.get("input_tokens", 0)
        output_tokens = usage.get("output_tokens", 0)
        total_tokens = usage.get("total_tokens", input_tokens + output_tokens)

        input_cost = input_tokens * INPUT_PRICE
        output_cost = output_tokens * OUTPUT_PRICE

        results.append({
            "User Query": query,
            "Teacher Response": reference,
            "GeneratedAnswer": response.text.strip(),
            "RetrievedContext": retrieval_context,
            "RetrievedPages": pages,
            "RetrievalLatency": retrieval_latency,
            "GenerationLatency": generation_latency,
            "Latency": total_latency,
            "InputTokens": input_tokens,
            "OutputTokens": output_tokens,
            "TotalTokens": total_tokens,
            "InputCost": input_cost,
            "OutputCost": output_cost,
            "TotalCost": input_cost + output_cost,
        })

  except Exception as e:
    print(f"\nError for question: {query}\n{e}")
    results.append({
            "User Query": query, "Teacher Response": reference,
            "GeneratedAnswer": "", "RetrievedContext": [], "RetrievedPages": [],
            "RetrievalLatency": np.nan, "GenerationLatency": np.nan, "Latency": np.nan,
            "InputTokens": np.nan, "OutputTokens": np.nan, "TotalTokens": np.nan,
            "InputCost": np.nan, "OutputCost": np.nan, "TotalCost": np.nan, })


results_df = pd.DataFrame(results)

display(results_df)

RAG generation:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:4109: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


content=[{'type': 'text', 'text': 'Based on the provided context, the purchases of property, plant and equipment (PP&E), which represent capital spending, were $(1,577) million for 2018 (or $1,577 million).', 'extras': {'signature': 'EpYBCpMBAWkUfRPU9IyWIZAS8aTLE65vKnaT9YlgHyRsKH8okPdY7eBntr9uRE2kNCjXsN0ls2lkf2s41GBIDpaBqacrm2mgr2wpJotNX2oyLrVfD59AEFPILveGMVJ3aw7UlZGfh/MyjIo0YG5hWMDJnNtK7pOs+4wGFPu/N+3UlQtUMDVKR9ddXgvVl6rMf8tCKQE2DJgc'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'normalized_finish_reason': 'stop', 'model_provider': 'google_genai'} id='lc_run--01a21555-aff3-729d-a26c-600b09f40801-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1314, 'output_tokens': 49, 'total_tokens': 1363, 'input_token_details': {'cache_read': 0}}


,User Query,Teacher Response,GeneratedAnswer,RetrievedContext,RetrievedPages,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is the FY2018 capital expenditure amount ...,$1577.00,"Based on the provided context, the purchases o...",[“Cash Flows from Investing Activities” sectio...,"[48, 59, 46]",0.303396,0.923506,1.228783,1314,49,1363,0.000394,0.000123,0.000517


# STAGE 5 - LLM EVALUATION - ANSWER QUALITY

In [50]:
import os
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

# Ensure environment variables are set before measuring
os.environ["GEMINI_API_KEY"] = api_key
os.environ["GOOGLE_API_KEY"] = api_key

quality_scores = []

for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Answer quality"):
    query = row["User Query"]
    generated = row["GeneratedAnswer"]
    reference = row["Teacher Response"]

    if not generated:
        quality_scores.append([np.nan] * 3)
        continue

    test_case = LLMTestCase(input=query, actual_output=generated, expected_output=reference)

    scores = []

    for metric in [correctness_metric, answer_relevancy_metric, helpfulness_metric]:
        try:
            metric.measure(test_case)
            scores.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            scores.append(np.nan)

    quality_scores.append(scores)

results_df[ ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"] ] = pd.DataFrame(quality_scores, index=results_df.index)
results_df["AnswerQuality"] = results_df[ ["AnswerCorrectness", "AnswerRelevancy", "Helpfulness"]].mean(axis=1)
display(results_df[ ["User Query", "AnswerCorrectness", "AnswerRelevancy", "Helpfulness", "AnswerQuality"] ])

Answer quality:   0%|          | 0/1 [00:00<?, ?it/s]

Output()

Output()

Output()

,User Query,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality
0,What is the FY2018 capital expenditure amount ...,0.7,1.0,1.0,0.9


# STAGE 6 - LLM EVALUATION - SIMILARITY METRICS

In [52]:
similarity_scores = []

for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Similarity metrics"):
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]

    if not generated:
        similarity_scores.append([np.nan] * 5)
        continue

    try:
        _, _, f1 = bert_scorer.score([generated], [reference] )
        bert = float(f1[0])
    except Exception:
        bert = np.nan

    try:
        rouge_scores = rouge.score(reference, generated)

        rouge1 = rouge_scores["rouge1"].fmeasure
        rouge2 = rouge_scores["rouge2"].fmeasure
        rougeL = rouge_scores["rougeL"].fmeasure
    except Exception:
        rouge1 = rouge2 = rougeL = np.nan

    try:
        meteor = meteor_score([reference.split()], generated.split() )
    except Exception:
        meteor = np.nan

    similarity_scores.append([bert, rouge1, rouge2, rougeL, meteor])


similarity_columns = ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR" ]
results_df[similarity_columns] = pd.DataFrame(similarity_scores, index=results_df.index )
display(results_df[["User Query"] + similarity_columns])

Similarity metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is the FY2018 capital expenditure amount ...,0.806147,0.0,0.0,0.0,0.0


# STAGE 7 - LLM EVALUATION - RAG EVALUATION METRICS

In [56]:
async def calculate_ragas(query, reference, generated):
    try:
        score = await answer_correctness_metric.ascore(user_input=query, response=generated, reference=reference)
        return float(score.value)
    except Exception as e:
        print("RAGAS error:", e)
        return np.nan


rag_scores = []

for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc="RAG evaluation"):
    query = row["User Query"]
    reference = row["Teacher Response"]
    generated = row["GeneratedAnswer"]
    context = row["RetrievedContext"]

    if not generated:
        rag_scores.append([np.nan] * 6)
        continue

    # RAGAS
    ragas_score = await calculate_ragas(query, reference, generated)

    # DEEPEVAL RAG METRICS
    test_case = LLMTestCase(input=query, actual_output=generated, expected_output=reference, retrieval_context=context )

    metric_values = []

    for metric in [ context_precision_metric, context_recall_metric, faithfulness_metric, rag_answer_relevancy_metric, rag_geval_metric ]:
        try:
            metric.measure(test_case)
            metric_values.append(float(metric.score))
        except Exception as e:
            print(f"{metric.__class__.__name__} error: {e}")
            metric_values.append(np.nan)

    # Order: RAGAS, precision, recall, faithfulness,
    # answer relevancy, GEval
    rag_scores.append([ragas_score, *metric_values])


rag_columns = [
    "Ragas AnswerCorrectness",
    "Context Precision",
    "Context Recall",
    "Faithfulness",
    "RAG Answer Relevancy",
    "RAG GEval"
]

results_df[rag_columns] = pd.DataFrame(rag_scores, index=results_df.index )
results_df["RAG Evaluation Score"] = results_df[rag_columns].mean(axis=1)
display(results_df[["User Query"] + rag_columns + ["RAG Evaluation Score"]])

RAG evaluation:   0%|          | 0/1 [00:00<?, ?it/s]

Output()

RAGAS error: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-3-small is not found for API version v1, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


Output()

Output()

Output()

Output()

,User Query,Ragas AnswerCorrectness,Context Precision,Context Recall,Faithfulness,RAG Answer Relevancy,RAG GEval,RAG Evaluation Score
0,What is the FY2018 capital expenditure amount ...,NaN,1.0,1.0,1.0,1.0,0.9,0.98


# STAGE 8 - LLM EVALUATION - OPERATIONAL METRICS SUMMARY

In [57]:
operational_columns = [
    "RetrievalLatency",
    "GenerationLatency",
    "Latency",
    "InputTokens",
    "OutputTokens",
    "TotalTokens",
    "InputCost",
    "OutputCost",
    "TotalCost"
]

operational_summary = pd.DataFrame({
    "Metric" : operational_columns,
    "Average": [ results_df[c].mean() for c in operational_columns],
    "Total"  : [results_df[c].sum() for c in operational_columns] })

display(operational_summary.round(6))

,Metric,Average,Total
0,RetrievalLatency,0.303396,0.303396
1,GenerationLatency,0.923506,0.923506
2,Latency,1.228783,1.228783
3,InputTokens,1314.000000,1314.000000
4,OutputTokens,49.000000,49.000000
5,TotalTokens,1363.000000,1363.000000
6,InputCost,0.000394,0.000394
7,OutputCost,0.000123,0.000123
8,TotalCost,0.000517,0.000517


# STAGE 9 - FINAL EVALUATION SUMMARY

In [58]:
# ROUND ALL METRIC COLUMNS

metric_columns = [
    "AnswerCorrectness",
    "AnswerRelevancy",
    "Helpfulness",
    "AnswerQuality",
    *similarity_columns,
    *rag_columns,
    "RAG Evaluation Score"
]

results_df[metric_columns] = results_df[metric_columns].round(3)


# FINAL QUESTION-WISE TABLE

final_columns = [
    "User Query",
    "Teacher Response",
    "GeneratedAnswer",
    *metric_columns,
    *operational_columns
]

final_df = results_df[final_columns].copy()

display(final_df)


# AVERAGE EVALUATION SUMMARY

summary_df = pd.DataFrame({
    "Metric": metric_columns,
    "Average Score": [ results_df[c].mean() for c in metric_columns]
})

summary_df["Average Score"] = (summary_df["Average Score"].round(3) )
display(summary_df)


# SAVE OUTPUTS

final_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
summary_df.to_csv("RAG_LLM_EVALUATION_SUMMARY.csv", index=False, encoding="utf-8-sig" )
operational_summary.to_csv("RAG_LLM_OPERATIONAL_SUMMARY.csv", index=False, encoding="utf-8-sig")

print("Evaluation completed.")
print("Final results:", OUTPUT_CSV)
print("Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv")
print("Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv")

,User Query,Teacher Response,GeneratedAnswer,AnswerCorrectness,AnswerRelevancy,Helpfulness,AnswerQuality,BERT,ROUGE-1,ROUGE-2,...,RAG Evaluation Score,RetrievalLatency,GenerationLatency,Latency,InputTokens,OutputTokens,TotalTokens,InputCost,OutputCost,TotalCost
0,What is the FY2018 capital expenditure amount ...,$1577.00,"Based on the provided context, the purchases o...",0.7,1.0,1.0,0.9,0.806,0.0,0.0,...,0.98,0.303396,0.923506,1.228783,1314,49,1363,0.000394,0.000123,0.000517


,Metric,Average Score
0,AnswerCorrectness,0.700
1,AnswerRelevancy,1.000
2,Helpfulness,1.000
3,AnswerQuality,0.900
4,BERT,0.806
5,ROUGE-1,0.000
6,ROUGE-2,0.000
7,ROUGE-L,0.000
8,METEOR,0.000
9,Ragas AnswerCorrectness,NaN


Evaluation completed.
Final results: FINANCEBENCH_GEMINI_EVALUATION.csv
Evaluation summary: RAG_LLM_EVALUATION_SUMMARY.csv
Operational summary: RAG_LLM_OPERATIONAL_SUMMARY.csv
